In [1]:
!uv add gitsource

Resolved 129 packages in 4ms


Checked 125 packages in 4ms


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
lesson_count = len(files)
print(lesson_count)

72


In [6]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

In [9]:
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [10]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-18 19:04:10--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.2’

rag_helper.py.2     100%[===================>]   2.08K  --.-KB/s    in 0.002s  

2026-06-18 19:04:10 (891 KB/s) - ‘rag_helper.py.2’ saved [2134/2134]



In [11]:
from rag_helper import CustomRAG

In [12]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
assistant = CustomRAG(index=index, llm_client=openai_client, model="gpt-5.4-mini")

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
##input_tokens=7136
#Output tokens: 90


 Input tokens: 7136
Output tokens: 135


The loop keeps calling the model by checking whether the model returned any `function_call` items.

- After each API call, the code sets `has_function_calls = False`
- It processes `response.output`
- If it finds a `function_call`, it runs the tool, appends the tool result, and sets `has_function_calls = True`
- At the end of the turn, if `has_function_calls == False`, it breaks out of the `while True` loop

So the model is called again only if it asked for a tool on the previous turn. If it returns a normal message with no function calls, the loop stops.


In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [16]:
print(chunks[0])

{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone

In [17]:
from minsearch import Index

chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename', 'start']
)
chunk_index.fit(chunks)

In [18]:
rag_chunked = CustomRAG(index=chunk_index, llm_client=openai_client, model="gpt-5.4-mini")

answer= rag_chunked.rag( "How does the agentic loop keep calling the model until it stops?")

print(answer)

##input_tokens=2319



 Input tokens: 2319
Output tokens: 89


The loop keeps calling the model with a `while True` loop. After each model response, it checks whether any `function_call` items were returned.

- If there are function calls, it executes them, appends the tool outputs to the message history, and loops again.
- If there are no function calls, it breaks and stops.

So the stopping condition is: **no function calls in the current turn**.


In [19]:
!uv add toyaikit

Resolved 129 packages in 0.71ms
Checked 125 packages in 1ms


In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from typing import List, Dict, Any

call_count = 0

def search_chunks(query: str) -> List[Dict[str, Any]]:
    """
    Search the course lesson pages for information relevant to the query.
    """
    global call_count
    call_count += 1
    return chunk_index.search(
        query,
        num_results=5
    )

agent_instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()



In [21]:
agent_tools = Tools()
agent_tools.add_tool(search_chunks)

In [22]:
openai_client = OpenAIClient(model="gpt-5.4-mini", client= OpenAI())

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    llm_client=openai_client,
    chat_interface=None,
)

In [23]:
call_count = 0

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=None,
)

In [24]:
print(result.last_message)
print()
print(f"search_chunks() called {call_count} times")

The **agentic loop** is a repeated cycle where the model and tools take turns until the model is satisfied.

In the course, it works like this:

1. Send the user question plus the conversation history to the LLM.
2. The LLM may return a **function/tool call** instead of a final answer.
3. Your code executes that tool call, like `search`.
4. You append the tool result back into the message history.
5. You call the LLM again with the updated history.
6. Repeat until the LLM returns a final answer with **no more tool calls**.

So the loop is basically: **LLM decides → tool runs → result वापस to LLM → repeat**.

## How it differs from plain RAG

**Plain RAG** is usually a one-shot pipeline:

```python
def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    return llm(prompt)
```

That means:

- you retrieve once,
- build one prompt,
- make one LLM call,
- done.

## Main difference

- **Plain RAG**: retrieval happens once, before gene